# ProductIQ Phase 3 — Step 3.11: Representation Dataset Generation

Materialize Phase 3 product representations from the canonical catalog (sample run).

In [ ]:
from pathlib import Path

import pandas as pd
from data.export.schema import PROCESSED_PARQUET_FILENAME
from productiq.representation import (
    generate_representation_dataset,
    load_representation_dataset_manifest,
    validate_representation_dataset,
)
from productiq.representation.dataset_schema import REPRESENTATION_DATASET_COLUMN_ORDER


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src").exists():
            return candidate
    msg = "Could not locate ProductIQ project root"
    raise RuntimeError(msg)


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
PROCESSED_DIR = PROJECT_ROOT / "resources" / "processed"
SOURCE = PROCESSED_DIR / PROCESSED_PARQUET_FILENAME
OUTPUT = PROCESSED_DIR / "product_representations_sample.parquet"
MANIFEST = PROCESSED_DIR / "product_representations_sample.manifest.json"

In [ ]:
sample_source = PROCESSED_DIR / "_notebook_catalog_sample.parquet"
canonical = pd.read_parquet(SOURCE).head(50)
canonical.to_parquet(sample_source, index=False)

result = generate_representation_dataset(
    sample_source,
    OUTPUT,
    manifest_path=MANIFEST,
    batch_size=10,
    validate_canonical_input=False,
)
print(result.report.summary())
validation = validate_representation_dataset(OUTPUT, expected_row_count=50)
print(validation.summary())
print("columns:", len(REPRESENTATION_DATASET_COLUMN_ORDER))

In [ ]:
manifest = load_representation_dataset_manifest(MANIFEST)
print({key: manifest[key] for key in ("row_count", "schema_version", "checksum", "source_row_count")})
pd.read_parquet(OUTPUT)[["product_id", "product_text", "lexical_text"]].head(3)